# Tabelas, Views, Materialized e Temporary Views

### O que são, quais a diferenças e quando usar cada uma delas


### Tabelas 
É um conjunto de dados estruturados que são armazenados fisicamente em um local (armazenamento)

**Quando usar:** Use para armazenar de forma permanente os dados brutos ou tratados do seu negócio. Com ela, você armazena os dados permanentemente e suportam manipulações por meio de comandos DML 

### Views
É uma tabela virtual que não armazena os dados fisicamente, ela é gerada a partir de uma consulta da tabela base e processa os dados toda vez que há uma nova requisição, é como se ela rodasse o comando que gera ela a partir da tabela base, e depois rodasse o comando que foi sua consulta a partir dessa primeira requisição. 

**Quando usar**: Ideal para simplificar consultar muito complexas, encapsular regras de negocio ou filtrar informações sensíveis sem precisar duplicar os dados da base. É uma excelente forma de disponibilizar uma interface limpa de consumo (somente os dados necessários) para os usuários. 

### Temporary View
É uma view com ciclo de vida restrito e efêmero. Em um pipeline como o Lakeflow Spark Declarative Pipeline, ela existe somente durante sua execução, e não pode ser acessada por fora dele, se tornando restrita ao pipeline. 

**Quando usar:** Use para criar passos intermediários de transformação ou realizar verificações de lógica sem gastar recursos de armazenamento ou poluir o catalog com um monte de views/caminhos inúteis. 

### Materialized View 
É uma view declarativa que pré-calcula e armazena os dados fisicamente em cache. Ao contrário da view tradicional, a materialized view rastreia as alterações da base e atualiza com base em agendamentos. **Vantagem**: Realiza calculo de forma incremental, processando apenas as mudanças recentes ao invés de processar a tabela inteira. 

**Quando usar**: Use para transformações custosas, joins complexos e cálculos de agregações lidos frequentementes. Ela é a escolha perfeita para a camada Gold para ser consumida pelo BI, relatórios e Dashboards pois o usuário acessará dados que já estão pré-calculados garantindo uma baixa latência de consulta sem o custo de rodar as agregações em tempo real

Usaremos aqui a tabela gerada pelo notebook **Data Lakehouse....** , caso necessite gerar a tabela, rode as células encontradas no outro notebook. Iremos agora explorar a criacão de Views e CTAS a partir da tabela criada anteriormente


In [0]:
%sql 

DESCRIBE HISTORY workspace.default.vendas

## CREATE TABLE AS SELECT (CTAS)

CTAS é o termo que se refere ao comando SQL 'CREATE TABLE AS SELECT', ele serve para criar uma tabela a partir de uma consulta de outra tabela, com isso, você consegue pular a etapa de descrever o schema da tabela, passando como referência outra tabela. 

Em palavras mais fáceis, é um comando de cópia de tabela, basicamente o que o CTAS faz é criar a cópia de uma tabela a partir de uma query que pode ser qualquer coisa 

Sua estrutura padrão é: 

`CREATE TABLE <catalog.schema.tabela> AS 
SELECT * FROM <catalog.schema.tabela_base>`

In [0]:
%sql
-- Criando uma tabela usando CTA a partir de uma versão passada da tabela vendas 
CREATE TABLE workspace.default.vendas_version_9 AS 
SELECT * FROM Workspace.default.vendas VERSION AS OF 9; 

In [0]:
%sql
SELECT * FROM workspace.default.vendas_version_9; 

Há também um cenário onde não queremos criar uma cópia da tabela, e sim fazer um fallback dela com uma versão anterior, para isso usamos o comando `INSERT OVERWRITE TABLE` que nos permite sobreescrever a tabela com versões passadas dela ou de outra tabela, como é o caso do exemplo abaixo, para não perdermos nossa base eu quero sobreescrever a tabela que criamos na célula acima, com dados de outra versão da tabela base

In [0]:
%sql
-- substitui toda a tabela pelos novos dados
INSERT OVERWRITE TABLE vendas_version_9
SELECT * FROM vendas VERSION AS OF 4;

In [0]:
%sql
DESCRIBE HISTORY vendas_version_9; 

Trabalhando com VIEWs

In [0]:
%sql
-- Criando uma view a partir de uma versão da tabela
CREATE OR REPLACE VIEW vendas_view AS 
SELECT * FROM workspace.default.vendas VERSION AS OF 11

In [0]:
%sql
Select * from vendas_view; 

Outras formas de consultar ou restaurar versões passadas da tabela 

In [0]:
%sql
-- Outra forma de consultar uma versão da tabela é usando o operador @ 
select * from vendas @V5; 

In [0]:
%sql 
-- Caso não lembre qual é a versão correta da tabela que quer consultar, mas lembra o dia que ela teve a última alteração, você pode consultar a tabela passando o timestamp da atualização 

SELECT * FROM vendas TIMESTAMP AS OF '2026-04-12T01:56:38.000+00:00'; 

Agora o próximo cenário, imagina que por algum motivo deletou a tabela por engano e precisa restaurar uma versão antiga dela, há o comando de cópia, criar uma tabela a partir da versão dela, pode usar o INSERT OVERWRITE TABLE para sobreescrever a tabela com uma versão antiga dela ou há também o comando RESTORE TABLE, abaixo vamos explorar ele e o que acontece com o DESCRIBE HISTORY ao usar o comando RESTORE 

spoiler: Você terá mais dois registros no DESCRIBE, um referente ao DELETE e outro REFERENTE ao RESTORE, ou seja, o log nunca é apagado, por mais que você restore para uma versão anterior, ele armazena esse registro também. 

In [0]:
%sql
DELETE FROM vendas_version_9; 

In [0]:
%sql
-- RESTAURAR DADOS - VERSÃO
RESTORE TABLE vendas_version_9 VERSION AS OF 2

In [0]:
%sql 
DESCRIBE HISTORY vendas_version_9; 